<a href="https://colab.research.google.com/github/jaalvalcan/GEE_index_sets/blob/main/flood.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install earthengine-api xarray xee geemap matplotlib -q

In [ ]:
import ee
import geemap
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# 1. Authenticate and Initialize Earth Engine
try:
    ee.Initialize(project='ee-jaalvalcan') # Replace with your GEE Cloud Project ID if required
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-jaalvalcan')

In [ ]:


# 2. Define Area of Interest (AOI) - Massive floodplain near Eldorado do Sul, Brazil
# Coordinates: Longitude -51.32, Latitude -30.01
aoi = ee.Geometry.Point([-51.32, -30.01]).buffer(20000) # 20km buffer to capture wide valleys

# 3. Time Windows around the major flood surge
before_start, before_end = '2026-04-01', '2026-04-20'
after_start, after_end   = '2026-05-05', '2026-05-25'

# 4. Load Sentinel-1 Ground Range Detected (GRD)
s1_col = ee.ImageCollection("COPERNICUS/S1_GRD")

def get_sar_image(start_date, end_date, aoi):
    """Filters Sentinel-1 collection and returns a smoothed median image."""
    dataset = s1_col.filterBounds(aoi)\
                    .filterDate(start_date, end_date)\
                    .filter(ee.Filter.eq('instrumentMode', 'IW'))\
                    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
                    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))

    # Smooth out noise speckles using a focal mean
    raw_img = dataset.mean().clip(aoi)
    smoothed_img = raw_img.focal_mean(30, 'circle', 'meters')
    return smoothed_img.select('VV')

print("Fetching SAR assets from Earth Engine...")
sar_before = get_sar_image(before_start, before_end, aoi)
sar_after = get_sar_image(after_start, after_end, aoi)

# Combine arrays side-by-side into a single image asset
combined_img = ee.Image.cat([sar_before.rename('before'), sar_after.rename('after')])

# 5. Extract to Numpy / Xarray using geemap
print("Downloading regional array data via Geemap...")
np_array = geemap.ee_to_numpy(combined_img, region=aoi, scale=40) # 40m scale for efficient memory processing

# Extract the channels
before_data = np_array[:, :, 0]
after_data = np_array[:, :, 1]

# Rebuild into a clean, local Xarray Dataset manually
ds = xr.Dataset(
    data_vars={
        'before': (['y', 'x'], before_data),
        'after': (['y', 'x'], after_data),
    }
)

# Mask out missing edge/padding data
ds = ds.where(ds['before'] != 0)

# Calculate Flood Difference (Drop in Backscatter)
# High positive values = Massive reduction in radar bounce (Land turning into smooth water)
ds['flood_diff'] = ds['before'] - ds['after']

print("Data successfully converted. Building statistical plots...")

# 6. Plotting Imagery and Histogram Analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('SAR High-Impact Flood Detection - Rio Grande do Sul, Brazil', fontsize=16, weight='bold')

# Plot Before
ds['before'].plot(ax=axes[0, 0], cmap='gray', vmin=-25, vmax=0)
axes[0, 0].set_title('Pre-Flood SAR Backscatter (Dry Land/Rivers)')
axes[0, 0].axis('off')

# Plot After
ds['after'].plot(ax=axes[0, 1], cmap='gray', vmin=-25, vmax=0)
axes[0, 1].set_title('Post-Flood SAR Backscatter (Dark Inundated Basins)')
axes[0, 1].axis('off')

# Plot Difference Map
ds['flood_diff'].plot(ax=axes[1, 0], cmap='Blues', vmin=0, vmax=12)
axes[1, 0].set_title('Flood Change Map (Drop in dB)')
axes[1, 0].axis('off')

# Plot Statistical Histogram
diff_values = ds['flood_diff'].values.flatten()
diff_values = diff_values[~np.isnan(diff_values)]

axes[1, 1].hist(diff_values, bins=100, color='dodgerblue', edgecolor='black', alpha=0.7)
axes[1, 1].axvline(x=5.0, color='red', linestyle='--', linewidth=2, label='Confirmed Inundation Threshold')
axes[1, 1].set_title('Statistical Backscatter Drop Histogram')
axes[1, 1].set_xlabel('Backscatter Decrease (Before - After in dB)')
axes[1, 1].set_ylabel('Pixel Count')
axes[1, 1].grid(True, linestyle=':', alpha=0.6)
axes[1, 1].legend()

plt.tight_layout()
plt.show()